# Phase 7 (PCR) - Model 4 + LoRMA (perm)
Dense retrieval with the frozen Phase-1 Model 4 (GQA and full-MHA layers, Mixture-of-Experts feed-forward) + hand-written LoRMA (perm) on the attention q/k/v/o and every MoE expert gate/up/down layer (routed and shared); the router stays frozen. Siamese encoder (query and candidate share the adapter), multi-positive contrastive loss, hard + in-batch negatives, retrieval over the whole candidate pool.

**Attach as inputs:**
1. The Phase-7 data-prep output (`tokenized_pcr.pkl`)
2. Your Phase-1 Model 4 folder (`legal-gqamoe-final`, contains `config.json` and weights)
3. The three code files `model.py`, `moe.py`, `legal_gqa_model.py` (they must sit in one folder)

**Output:** `/kaggle/working/pcr_m4_lorma/` (adapter weights, dev sweep table, `prediction.json`, `submission.zip`, `run_summary.json`)

**How the data insights are used (all decided from the data-prep bundle or from dev, nothing is hard-wired):**
- Documents are far longer than the context, so each case is represented by a fixed budget of chunks. The chunk view (head / tail / head+tail / full) is picked from the data-prep view analysis (`DOC_VIEW = "auto"`).
- Every query is also present in its own candidate pool, so the query's own id is masked in training negatives and in ranking.
- The number of relevant cases per query varies from 1 to 90+, so the loss is multi-positive (positives never compete with each other) and false negatives (other gold of the same query) are masked.
- Almost every pool document is gold for some query, so other queries' positives are natural hard negatives (in-batch), and score normalisation across queries (CSLS / dual softmax) is tried at inference. Dev decides which one is used.
- Dev is a separate set with its own pool, so it is the validation set (the train folds are one giant connected group and are not used).

## 1. Configuration

In [ ]:
import os
import sys
import json
import glob
import math
import time
import random
import shutil
import pickle
import zipfile

import numpy as np

# Env vars are only for local testing. On Kaggle leave them alone.
DATA_DIR = os.environ.get("PCR_DATA_DIR", None)
MODEL_DIR = os.environ.get("PCR_MODEL_DIR", None)
MODEL_PY = os.environ.get("PCR_MODEL_PY", None)
OUT_DIR = os.environ.get("PCR_OUT_DIR", "/kaggle/working/pcr_m4_lorma")

MODEL_NAME_HINT = "legal-gqamoe"  # folder name fragment of the Phase-1 Model 4 folder
MODEL_TYPE = "legal_gqa_mha_moe"  # model_type written in its config.json

SMOKE_TEST = False                # True = tiny run to check the code end to end
if os.environ.get("PCR_SMOKE") == "1":
    SMOKE_TEST = True

SEED = 42
RUN_TRAIN = True
RUN_TEST = True
RESUME = True                     # pick up a checkpoint from an attached earlier run
TRAIN_ON_DEV = False              # True = also train on dev queries (then there is no validation, last epoch is used)

# Document representation
DOC_VIEW = "auto"                 # "auto" (from the data-prep view analysis), "head", "tail", "headtail" or "full"
HEAD_CHUNKS = 1
TAIL_CHUNKS = 1
FULL_MAX_CHUNKS = 8               # only for DOC_VIEW = "full"
CHUNK_LEN = 1024                  # tokens per forward pass, including the closing SEP token
POOLING = "mean"                  # "mean" over the chunk tokens or "last" (the closing SEP token)
USE_PROJ = True                   # trainable d x d projection, starts as the identity
TEMPERATURE = 0.05

# Adapter
LORMA_VARIANT = "perm"             # "perm" (LoRMA-pi: dense d x d rotation per layer, slower) or "plus" (LoRMA+, cheap)
LORMA_R = 16
LORMA_ALPHA = 16                  # only used by plus (scale = alpha / r)
LORMA_DROPOUT = 0.05              # only used by plus
LR = 5e-4                         # adapter learning rate
PROJ_LR = 1e-4                    # projection learning rate

# Training
QUERIES_PER_STEP = 8
POS_PER_QUERY = 3                 # positives sampled per query per step (gold sets are much larger for some queries)
HARD_NEG_PER_QUERY = 3
HARD_NEG_RANGE = (3, 40)          # hard negatives come from this slice of the BM25 non-gold list (skips the very top, which may be unlabeled relevant)
EPOCHS = 5
WEIGHT_DECAY = 0.0
WARMUP_FRAC = 0.05
GRAD_CLIP = 1.0
CHUNK_MICRO_BATCH = 8             # chunks per forward pass during training
EVAL_MICRO_BATCH = 16
USE_CHECKPOINT = True             # gradient checkpointing, needed on a 16 GB T4
SAVE_EVERY_STEPS = 25
LOG_EVERY_STEPS = 5
SESSION_CAP_HOURS = 10.5          # the run saves and stops, and resumes on the next run

# Inference: everything below is chosen on dev
TOP_K = 7
NORM_GRID = [("none", 0), ("csls", 5), ("csls", 10), ("dsm", 0.02), ("dsm", 0.05), ("dsm", 0.1)]
HYBRID_LAMBDAS = [0.0, 0.25, 0.5, 1.0, 2.0]    # weight of token-BM25 (head+tail) added to the neural score
BM25_VIEW_TOKENS = 1280
BM25_K1 = 1.5
BM25_B = 0.75

SMOKE_TRAIN_QUERIES = 24
if SMOKE_TEST:
    CHUNK_LEN = 128
    EPOCHS = 2
    QUERIES_PER_STEP = 4
    CHUNK_MICRO_BATCH = 8
    SAVE_EVERY_STEPS = 3
    LOG_EVERY_STEPS = 2
    print("SMOKE_TEST is on: tiny data, 2 epochs, 128-token chunks, to check the code only.")

CONTENT_LEN = CHUNK_LEN - 1       # one position is kept for the closing SEP

os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
print("Config loaded.")

## 2. Find the inputs

In [ ]:
def find_one_file(roots, name):
    hits = []
    for root in roots:
        for dir_path, dir_names, file_names in os.walk(root):
            if name in file_names:
                hits.append(os.path.join(dir_path, name))
    if len(hits) == 0:
        return None
    hits.sort(key=len)
    return hits[0]


def find_model_py():
    # model.py is a very common name, so only accept a folder that also has moe.py and legal_gqa_model.py
    for root in ["/kaggle/input", "/kaggle/working"]:
        for dir_path, dir_names, file_names in os.walk(root):
            if "model.py" in file_names and "moe.py" in file_names and "legal_gqa_model.py" in file_names:
                return os.path.join(dir_path, "model.py")
    return None


def find_model_dir(name_hint):
    found = []
    for dir_path, dir_names, file_names in os.walk("/kaggle/input"):
        low = dir_path.lower()
        if "config.json" not in file_names or "checkpoint" in low:
            continue
        model_type = ""
        try:
            with open(os.path.join(dir_path, "config.json")) as f:
                model_type = json.load(f).get("model_type", "")
        except Exception:
            model_type = ""
        if model_type == MODEL_TYPE or name_hint in low:
            found.append(dir_path)
    if len(found) == 0:
        return None
    for path in found:
        if "final" in path.lower():
            return path
    return found[0]


if DATA_DIR is None:
    BUNDLE_PATH = find_one_file(["/kaggle/input"], "tokenized_pcr.pkl")
else:
    BUNDLE_PATH = os.path.join(DATA_DIR, "tokenized_pcr.pkl")
if BUNDLE_PATH is None or not os.path.exists(BUNDLE_PATH):
    raise FileNotFoundError("tokenized_pcr.pkl not found. Attach the Phase-7 data-prep output (Add Input).")
print("data bundle:", BUNDLE_PATH)

if MODEL_DIR is None:
    MODEL_DIR = find_model_dir(MODEL_NAME_HINT)
    if MODEL_DIR is None:
        raise FileNotFoundError("No Phase-1 Model 4 folder (model_type legal_gqa_mha_moe) found. Attach legal-gqamoe-final.")
print("model folder:", MODEL_DIR)

if MODEL_PY is None:
    MODEL_PY = find_model_py()
    if MODEL_PY is None:
        raise FileNotFoundError("Model 4 code (model.py + moe.py + legal_gqa_model.py in one folder) not found. Attach it (see the header).")
print("model code:", MODEL_PY)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as torch_checkpoint

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.float16
USE_SCALER = USE_AMP
torch.manual_seed(SEED)
print("device:", DEVICE, "| fp16 autocast:", USE_AMP)

START_TIME = time.time()
SESSION_CAP_SEC = SESSION_CAP_HOURS * 3600.0
if os.environ.get("PCR_SESSION_CAP_SEC") is not None:
    SESSION_CAP_SEC = float(os.environ.get("PCR_SESSION_CAP_SEC"))

## 3. Load the data bundle and read the data insights

In [ ]:
with open(BUNDLE_PATH, "rb") as f:
    B = pickle.load(f)

SEP_ID = int(B["sep_id"])
PAD_ID = int(B["pad_id"])
VOCAB_SIZE = int(B["vocab_size"])
print("splits:", B["splits"], "| vocab:", VOCAB_SIZE, "| SEP id:", SEP_ID, "| PAD id:", PAD_ID)
if PAD_ID == SEP_ID:
    print("PAD and SEP share an id: padding is always masked by length, never by token id.")

GOLD = B["gold"]
GOLDSET = {}
for split in B["splits"]:
    if split == "test":
        continue
    for qid in GOLD[split]:
        GOLDSET[(split, qid)] = set(GOLD[split][qid])

# What the data says (printed so the choices below can be checked)
for split in ["train", "dev"]:
    if split not in B["splits"]:
        continue
    counts = []
    gold_union = set()
    for qid in GOLD[split]:
        counts.append(len(GOLD[split][qid]))
        for c in GOLD[split][qid]:
            gold_union.add(c)
    arr = np.array(counts)
    pool = len(B["pool_ids"][split])
    print("%s: %d queries | pool %d | gold per query mean %.1f median %.0f max %d | %.0f%% of the pool is gold for some query" % (
        split, len(arr), pool, arr.mean(), np.median(arr), arr.max(), 100.0 * len(gold_union) / pool))

view_results = B.get("view_results", {})
print("data-prep view analysis (BM25 Micro-F1@7 on dev):", view_results)
if DOC_VIEW == "auto":
    best_name = None
    best_score = -1.0
    for name in ["head", "tail", "headtail", "full"]:
        if name in view_results and view_results[name] > best_score:
            best_score = view_results[name]
            best_name = name
    if best_name is None:
        best_name = "headtail"
    DOC_VIEW = best_name
print("document view used:", DOC_VIEW, "| head chunks", HEAD_CHUNKS, "| tail chunks", TAIL_CHUNKS, "| chunk length", CHUNK_LEN)

## 4. Load the frozen Phase-1 Model 4

In [ ]:
import importlib.util
import inspect

sys.path.insert(0, os.path.dirname(MODEL_PY))
spec = importlib.util.spec_from_file_location("legal_moe_model", MODEL_PY)
legal_mod = importlib.util.module_from_spec(spec)
sys.modules["legal_moe_model"] = legal_mod
spec.loader.exec_module(legal_mod)

LM, load_info = legal_mod.LegalGQAMoEForCausalLM.from_pretrained(MODEL_DIR, output_loading_info=True)
missing_keys = []
for key in load_info.get("missing_keys", []):
    if key != "lm_head.weight":          # tied to the token embedding
        missing_keys.append(key)
print("Phase-1 weights loaded | missing keys:", len(missing_keys), "| unexpected keys:", len(list(load_info.get("unexpected_keys", []))))
if len(missing_keys) > 0:
    raise RuntimeError("Phase-1 weights were NOT fully loaded, missing: " + str(missing_keys[:5]))
if LM.config.vocab_size != VOCAB_SIZE:
    print("WARNING: model vocab", LM.config.vocab_size, "!= data vocab", VOCAB_SIZE)
if LM.config.vocab_size <= SEP_ID:
    raise RuntimeError("data / model vocab mismatch")

# lm_head is tied to the token embedding. If it was not in the checkpoint it must be tied explicitly.
# (lm_head is not used for retrieval, but the tie is kept so the model equals the Phase-1 model.)
if "lm_head.weight" in list(load_info.get("missing_keys", [])):
    LM.lm_head.weight = LM.wte.weight
    print("lm_head.weight was not in the checkpoint -> tied to the token embedding")

# The output layer must share its weights with the token embedding, otherwise lm_head would be random.
if LM.lm_head.weight.data_ptr() != LM.wte.weight.data_ptr():
    tie_gap = float((LM.lm_head.weight - LM.wte.weight).abs().max())
    if tie_gap > 0:
        raise RuntimeError("lm_head is not tied to the token embedding (gap %.4f); the Phase-1 weights were not restored" % tie_gap)
print("lm_head tied to the token embedding: ok")

LM.float()
LM = LM.to(DEVICE)
LM.eval()
for p in LM.parameters():
    p.requires_grad = False

N_BASE = sum(p.numel() for p in LM.parameters())
D_MODEL = int(LM.wte.weight.shape[1])
print("model:", type(LM).__name__, "| params: %.2fM | hidden size %d" % (N_BASE / 1e6, D_MODEL))

for attr in ["max_position_embeddings", "n_positions", "block_size", "max_seq_len", "context_length", "n_ctx"]:
    value = getattr(LM.config, attr, None)
    if isinstance(value, int) and value > 0:
        print("model context (%s): %d" % (attr, value))
        if CHUNK_LEN > value:
            raise RuntimeError("CHUNK_LEN %d is longer than the model context %d" % (CHUNK_LEN, value))
        break


def backbone_hidden(input_ids):
    # Same computation as the Phase-1 forward(), stopped before lm_head. Causal model, right padding is harmless.
    T = input_ids.shape[1]
    x = LM.drop(LM.wte(input_ids))
    cos, sin = LM._rope_slice(0, T, input_ids.device, x.dtype)
    for blk in LM.blocks:
        if torch.is_grad_enabled() and USE_CHECKPOINT:
            def run(h, blk=blk):
                return blk(h, cos, sin, None, True, None, False)[0]
            x = torch_checkpoint(run, x, use_reentrant=False)
        else:
            x = blk(x, cos, sin, None, True, None, False)[0]
    return LM.ln_f(x)


# My manual forward must agree with the model's own forward.
with torch.no_grad():
    probe = torch.tensor(np.random.randint(3, LM.config.vocab_size, size=(1, 48)), device=DEVICE)
    own_logits = LM.lm_head(backbone_hidden(probe)).float()
    ref_logits = LM(input_ids=probe).logits.float()
    agree = float((own_logits.argmax(-1) == ref_logits.argmax(-1)).float().mean())
    diff = float((own_logits - ref_logits).abs().max())
print("manual forward vs model forward: top-1 agreement %.3f | max logit diff %.4f" % (agree, diff))
assert agree > 0.9, "manual forward does not match the model forward"

## 5. Document views and the encoder
A case is cut into chunks of `CHUNK_LEN` tokens. Each chunk is run through the model, pooled, projected and normalised; a case embedding is the mean of its chunk embeddings.

In [ ]:
def chunk_starts(L, mode):
    CL = CONTENT_LEN
    if L <= CL:
        return [0]
    n_cover = int(math.ceil(L / CL))
    starts = []
    if mode == "full":
        if n_cover <= FULL_MAX_CHUNKS:
            for i in range(n_cover):
                starts.append(min(i * CL, L - CL))
        else:
            for s in np.linspace(0, L - CL, FULL_MAX_CHUNKS):
                starts.append(int(round(s)))
    else:
        n_head = 0
        n_tail = 0
        if mode in ["head", "headtail"]:
            n_head = HEAD_CHUNKS
        if mode in ["tail", "headtail"]:
            n_tail = TAIL_CHUNKS
        if n_cover <= n_head + n_tail:
            for i in range(n_cover):
                starts.append(min(i * CL, L - CL))
        else:
            for i in range(n_head):
                starts.append(i * CL)
            for j in range(n_tail):
                starts.append(max(L - (j + 1) * CL, 0))
    starts = sorted(set(starts))
    return starts


CHUNK_CACHE = {}


def get_chunks(spec):
    # spec = (split, "q" or "c", id)
    if spec in CHUNK_CACHE:
        return CHUNK_CACHE[spec]
    split, kind, doc_id = spec
    if kind == "q":
        ids = B["tokens"][split]["queries"][doc_id]
    else:
        ids = B["tokens"][split]["cands"][doc_id]
    chunks = []
    for s in chunk_starts(len(ids), DOC_VIEW):
        chunks.append(ids[s:s + CONTENT_LEN])
    CHUNK_CACHE[spec] = chunks
    return chunks


def make_batch(group):
    longest = 0
    for c in group:
        if len(c) > longest:
            longest = len(c)
    T = longest + 1
    ids = np.full((len(group), T), PAD_ID, dtype=np.int64)
    lengths = np.zeros(len(group), dtype=np.int64)
    for i, c in enumerate(group):
        L = len(c)
        ids[i, :L] = c.astype(np.int64)
        ids[i, L] = SEP_ID
        lengths[i] = L + 1
    return torch.from_numpy(ids).to(DEVICE), torch.from_numpy(lengths).to(DEVICE)


def pool_hidden(h, lengths):
    T = h.shape[1]
    if POOLING == "last":
        rows = torch.arange(h.shape[0], device=h.device)
        return h[rows, lengths - 1]
    positions = torch.arange(T, device=h.device).unsqueeze(0)
    mask = (positions < lengths.unsqueeze(1)).to(h.dtype)
    summed = (h * mask.unsqueeze(-1)).sum(1)
    return summed / lengths.unsqueeze(-1).to(h.dtype)


class Projection(nn.Module):
    # d x d linear map that starts as the identity, so training starts at the plain Phase-1 pooled representation
    def __init__(self, d):
        super().__init__()
        self.weight = nn.Parameter(torch.zeros(d, d))
        with torch.no_grad():
            for i in range(d):
                self.weight[i, i] = 1.0

    def forward(self, x):
        return x @ self.weight.t()


PROJ_MODULE = None
if USE_PROJ:
    PROJ_MODULE = Projection(D_MODEL).to(DEVICE)


def pooled_chunks(chunks, grad):
    n = len(chunks)
    order = list(range(n))
    if not grad:
        order.sort(key=lambda i: len(chunks[i]))      # similar lengths together, less padding
    micro = CHUNK_MICRO_BATCH if grad else EVAL_MICRO_BATCH
    pieces = []
    piece_index = []
    for start in range(0, n, micro):
        idx = order[start:start + micro]
        group = []
        for i in idx:
            group.append(chunks[i])
        ids, lengths = make_batch(group)
        with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            hidden = backbone_hidden(ids)
        pooled = pool_hidden(hidden.float(), lengths)
        if PROJ_MODULE is not None:
            pooled = PROJ_MODULE(pooled)
        pooled = F.normalize(pooled, dim=-1)
        pieces.append(pooled)
        for i in idx:
            piece_index.append(i)
    all_pooled = torch.cat(pieces, 0)
    inverse = torch.empty(n, dtype=torch.long)
    inverse[torch.tensor(piece_index, dtype=torch.long)] = torch.arange(n)
    return all_pooled[inverse.to(all_pooled.device)]


def embed_docs(specs, grad):
    chunks = []
    owner = []
    for k, spec in enumerate(specs):
        for c in get_chunks(spec):
            chunks.append(c)
            owner.append(k)
    chunk_emb = pooled_chunks(chunks, grad)
    owner_t = torch.tensor(owner, dtype=torch.long, device=chunk_emb.device)
    zeros = torch.zeros(len(specs), chunk_emb.shape[1], device=chunk_emb.device)
    summed = zeros.index_add(0, owner_t, chunk_emb)
    return F.normalize(summed, dim=-1)


def embed_split(split, max_queries=None):
    qids = list(B["query_ids"][split])
    if max_queries is not None:
        qids = qids[:max_queries]
    pids = list(B["pool_ids"][split])
    q_specs = []
    for q in qids:
        q_specs.append((split, "q", q))
    p_specs = []
    for p in pids:
        p_specs.append((split, "c", p))
    with torch.no_grad():
        Qe = embed_docs(q_specs, False).cpu()
        Pe = embed_docs(p_specs, False).cpu()
    return qids, pids, Qe, Pe


# how many chunks does the chosen view give per document?
for split in B["splits"]:
    n_q = []
    for q in B["query_ids"][split][:200]:
        n_q.append(len(get_chunks((split, "q", q))))
    n_c = []
    for p in B["pool_ids"][split][:200]:
        n_c.append(len(get_chunks((split, "c", p))))
    print("%s: chunks per query %.2f | per candidate %.2f (view %s)" % (split, np.mean(n_q), np.mean(n_c), DOC_VIEW))

## 6. Scoring, ranking and the dev-selected inference options
Plain cosine is always available. Optional steps (each chosen on dev): centring, CSLS / dual-softmax normalisation across queries, and a token-level BM25 (head+tail) hybrid.

In [ ]:
NEG_INF = float("-inf")


def make_scores(Qe, Pe, qids, pids, center):
    Q = Qe
    P = Pe
    if center:
        mu = torch.cat([Q, P], 0).mean(0, keepdim=True)
        Q = F.normalize(Q - mu, dim=-1)
        P = F.normalize(P - mu, dim=-1)
    S = Q @ P.t()
    pos_of = {}
    for i, p in enumerate(pids):
        pos_of[p] = i
    for r, q in enumerate(qids):
        if q in pos_of:
            S[r, pos_of[q]] = NEG_INF        # a query is in its own pool: never retrieve itself
    return S


def apply_norm(S, norm, param):
    if norm == "none":
        return S
    finite = torch.isfinite(S)
    if norm == "csls":
        low = S.masked_fill(~finite, -1.0)
        k = min(int(param), S.shape[0])
        r_cand = low.topk(k, dim=0).values.mean(0, keepdim=True)
        out = 2.0 * S - r_cand
        return out.masked_fill(~finite, NEG_INF)
    if norm == "dsm":
        z = S / float(param)
        out = torch.log_softmax(z, dim=1) + torch.log_softmax(z, dim=0)
        return out.masked_fill(~finite, NEG_INF)
    raise ValueError(norm)


def zscore_rows(S):
    finite = torch.isfinite(S)
    count = finite.sum(1, keepdim=True).clamp(min=1).to(S.dtype)
    filled = S.masked_fill(~finite, 0.0)
    mean = filled.sum(1, keepdim=True) / count
    centered = (filled - mean) * finite.to(S.dtype)
    var = (centered ** 2).sum(1, keepdim=True) / count
    out = (S - mean) / (var.sqrt() + 1e-6)
    return out.masked_fill(~finite, NEG_INF)


def top_ids(S, pids, k):
    k = min(k, S.shape[1])
    idx = S.topk(k, dim=1).indices
    out = []
    for r in range(idx.shape[0]):
        row = []
        for j in idx[r].tolist():
            row.append(pids[j])
        out.append(row)
    return out


def micro_prf(pred_lists, qids, gold, k):
    tp = 0
    n_gold = 0
    n_pred = 0
    for r, q in enumerate(qids):
        g = set(gold[q])
        n_gold += len(g)
        n_pred += k
        for c in pred_lists[r][:k]:
            if c in g:
                tp += 1
    p = tp / max(n_pred, 1)
    rec = tp / max(n_gold, 1)
    f1 = 0.0
    if p + rec > 0:
        f1 = 2 * p * rec / (p + rec)
    return p, rec, f1


def recall_at(S, qids, pids, gold, k):
    lists = top_ids(S, pids, k)
    tp = 0
    n_gold = 0
    for r, q in enumerate(qids):
        g = set(gold[q])
        n_gold += len(g)
        for c in lists[r]:
            if c in g:
                tp += 1
    return tp / max(n_gold, 1)


# ---- token-level BM25 over the head+tail token view (no tokenizer needed, terms are token ids)
import scipy.sparse as sp


def bm25_view(ids):
    half = BM25_VIEW_TOKENS // 2
    if len(ids) <= BM25_VIEW_TOKENS:
        return ids
    return np.concatenate([ids[:half], ids[-half:]])


def count_matrix(arrays):
    indptr = [0]
    indices = []
    data = []
    for ids in arrays:
        view = bm25_view(ids)
        view = view[view != SEP_ID]
        if len(view) == 0:
            indptr.append(indptr[-1])
            continue
        terms, counts = np.unique(view, return_counts=True)
        indices.append(terms.astype(np.int64))
        data.append(counts.astype(np.float32))
        indptr.append(indptr[-1] + len(terms))
    if len(indices) == 0:
        indices_all = np.zeros(0, dtype=np.int64)
        data_all = np.zeros(0, dtype=np.float32)
    else:
        indices_all = np.concatenate(indices)
        data_all = np.concatenate(data)
    return sp.csr_matrix((data_all, indices_all, np.array(indptr)), shape=(len(arrays), VOCAB_SIZE))


def bm25_scores(split, qids, pids):
    P_arrays = []
    for p in pids:
        P_arrays.append(B["tokens"][split]["cands"][p])
    Q_arrays = []
    for q in qids:
        Q_arrays.append(B["tokens"][split]["queries"][q])
    X = count_matrix(P_arrays).tocsr()
    n_docs = X.shape[0]
    df = np.bincount(X.indices, minlength=X.shape[1]).astype(np.float64)
    idf = np.log(1.0 + (n_docs - df + 0.5) / (df + 0.5))
    dl = np.asarray(X.sum(axis=1)).ravel().astype(np.float64)
    avgdl = max(dl.mean(), 1.0)
    row_of_nz = np.repeat(np.arange(n_docs), np.diff(X.indptr))
    tf = X.data.astype(np.float64)
    denom = tf + BM25_K1 * (1.0 - BM25_B + BM25_B * dl[row_of_nz] / avgdl)
    weights = idf[X.indices] * tf * (BM25_K1 + 1.0) / denom
    W = sp.csr_matrix((weights.astype(np.float32), X.indices.copy(), X.indptr.copy()), shape=X.shape)
    Q = count_matrix(Q_arrays).tocsr()
    Q.data = np.log1p(Q.data)
    S = torch.from_numpy((Q @ W.T).toarray().astype(np.float32))
    pos_of = {}
    for i, p in enumerate(pids):
        pos_of[p] = i
    for r, q in enumerate(qids):
        if q in pos_of:
            S[r, pos_of[q]] = NEG_INF
    return S


def run_sweep(Qe, Pe, qids, pids, gold, bm25_S):
    bm25_Z = zscore_rows(bm25_S)
    rows = []
    order = 0
    for center in [False, True]:
        S0 = make_scores(Qe, Pe, qids, pids, center)
        for norm, param in NORM_GRID:
            S1 = zscore_rows(apply_norm(S0, norm, param))
            for lam in HYBRID_LAMBDAS:
                if lam == 0.0:
                    S2 = S1
                else:
                    S2 = S1 + lam * bm25_Z
                    S2 = S2.masked_fill(~torch.isfinite(S1), NEG_INF)
                lists = top_ids(S2, pids, TOP_K)
                p, r, f1 = micro_prf(lists, qids, gold, TOP_K)
                rows.append({"order": order, "center": center, "norm": norm, "param": param, "lambda": lam,
                             "precision": p, "recall": r, "f1": f1})
                order += 1
    rows.sort(key=lambda d: (-d["f1"], d["order"]))      # ties keep the simpler option first
    return rows


def final_scores(Qe, Pe, qids, pids, cfg, bm25_S):
    S0 = make_scores(Qe, Pe, qids, pids, cfg["center"])
    S1 = zscore_rows(apply_norm(S0, cfg["norm"], cfg["param"]))
    if cfg["lambda"] == 0.0:
        return S1
    S2 = S1 + cfg["lambda"] * zscore_rows(bm25_S)
    return S2.masked_fill(~torch.isfinite(S1), NEG_INF)


def plain_f1(Qe, Pe, qids, pids, gold):
    S = make_scores(Qe, Pe, qids, pids, False)
    lists = top_ids(S, pids, TOP_K)
    return micro_prf(lists, qids, gold, TOP_K)

## 7. Zero-shot baseline on dev (frozen Phase-1 model, no adapter)
This is the "how well do Phase-1 representations transfer" row for the report, next to token-BM25.

In [ ]:
ZERO_SHOT = {}
if "dev" in B["splits"]:
    dev_gold = GOLD["dev"]
    t0 = time.time()
    saved_pooling = POOLING
    for pooling in ["mean", "last"]:
        POOLING = pooling
        qids, pids, Qe, Pe = embed_split("dev")
        p, r, f1 = plain_f1(Qe, Pe, qids, pids, dev_gold)
        S_center = make_scores(Qe, Pe, qids, pids, True)
        lists = top_ids(S_center, pids, TOP_K)
        pc, rc, f1c = micro_prf(lists, qids, dev_gold, TOP_K)
        ZERO_SHOT["frozen_" + pooling] = f1
        ZERO_SHOT["frozen_" + pooling + "_centered"] = f1c
        print("frozen Phase-1, %-4s pooling: Micro-F1@%d = %.4f (P %.4f, R %.4f) | centered: %.4f" % (pooling, TOP_K, f1, p, r, f1c))
    POOLING = saved_pooling
    qids = list(B["query_ids"]["dev"])
    pids = list(B["pool_ids"]["dev"])
    dev_bm25 = bm25_scores("dev", qids, pids)
    lists = top_ids(dev_bm25, pids, TOP_K)
    p, r, f1 = micro_prf(lists, qids, dev_gold, TOP_K)
    ZERO_SHOT["token_bm25_headtail"] = f1
    print("token-BM25 (head+tail %d tokens): Micro-F1@%d = %.4f (P %.4f, R %.4f)" % (BM25_VIEW_TOKENS, TOP_K, f1, p, r))
    print("word-level BM25 from the data-prep notebook (head+tail 1024 words):", view_results.get("headtail"))
    print("zero-shot dev evaluation took %.0fs" % (time.time() - t0))
else:
    print("No dev split: skipping the zero-shot baseline.")

## 8. Hand-written LoRMA (perm)

LoRMA changes the output side of a frozen layer multiplicatively. `W0` is frozen, `z = W0 x`.
- **perm:** `y = R z`, where `R` is `B A` with row `i` rotated by `i` places. It starts as the identity.
- **plus:** `y = z + (alpha / r) B (A z)`. Starts with `B = 0`.

`LORMA_VARIANT = "perm"` is the one asked for. In Phase 4, perm did not learn on Model 1 (loss stayed flat); if that happens here set `"plus"`.

In [ ]:
_ROT_CACHE = {}
_EYE_CACHE = {}


def hand_dropout(x, p, training):
    # dropout written by hand: zero some entries, scale the rest
    if (not training) or p == 0.0:
        return x
    keep_prob = 1.0 - p
    rand = torch.rand_like(x)
    mask = (rand < keep_prob).to(x.dtype)
    return x * mask / keep_prob


def hand_uniform_(tensor, bound):
    # fill a tensor with numbers drawn uniformly from [-bound, bound]
    with torch.no_grad():
        tensor.copy_(torch.rand_like(tensor) * 2.0 * bound - bound)


def get_rot_idx(d, device):
    # index table for the row rotation: entry (i, j) = (j - i) mod d
    key = (d, str(device))
    if key not in _ROT_CACHE:
        rows = torch.arange(d, device=device).unsqueeze(1)
        cols = torch.arange(d, device=device).unsqueeze(0)
        diff = cols - rows
        _ROT_CACHE[key] = diff % d
    return _ROT_CACHE[key]


def get_eye(d, device):
    # identity matrix built by hand: zeros with ones on the diagonal
    key = (d, str(device))
    if key not in _EYE_CACHE:
        eye = torch.zeros(d, d, device=device)
        diag = torch.arange(d, device=device)
        eye[diag, diag] = 1.0
        _EYE_CACHE[key] = eye
    return _EYE_CACHE[key]


def rotate_rows(M):
    # row i of M is shifted right by i places (wrapping around). Written with plain index tensors.
    d = M.shape[0]
    rows = torch.arange(d, device=M.device).unsqueeze(1).expand(d, d)
    cols = get_rot_idx(d, M.device)
    return M[rows, cols]


class LoRMALinear(nn.Module):
    # LoRMA: multiplicative update on the output side of the frozen layer. W0 is frozen, z = W0 x.
    #   plus: W = ((alpha / r) B A + I) W0      ->  y = z + (alpha / r) B (A z)
    #   perm: W = P(B A) W0                     ->  y = R z, R = row i of (B A) rotated by i places
    # Both start exactly at the Phase-1 weights (plus: B = 0; perm: R = I).
    # perm is computed as y = z + z (R - I)^T so the small learned part is not lost in fp16.
    def __init__(self, base, r, alpha, dropout, variant):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        d_out = base.weight.shape[0]
        self.d = d_out
        self.r = r
        self.variant = variant
        self.scale = alpha / r
        self.p_drop = dropout
        dev = base.weight.device
        self.lorma_A = nn.Parameter(torch.empty(r, d_out, device=dev, dtype=torch.float32))
        self.lorma_B = nn.Parameter(torch.zeros(d_out, r, device=dev, dtype=torch.float32))
        self.calls = 0                  # how often this layer ran (used by the activity check)
        self.reset_parameters()

    def reset_parameters(self):
        bound = 1.0 / math.sqrt(self.d)
        hand_uniform_(self.lorma_A, bound)
        with torch.no_grad():
            self.lorma_B.zero_()
            if self.variant == "perm":
                self.lorma_B[:, 0] = 1.0       # B A then has a single column of ones ...
                self.lorma_A[0].zero_()
                self.lorma_A[0, 0] = 1.0       # ... and the row rotation turns it into the identity

    def forward(self, x):
        self.calls += 1
        z = self.base(x)
        if self.variant == "plus":
            z32 = z.float()
            z32 = hand_dropout(z32, self.p_drop, self.training)
            low = z32 @ self.lorma_A.t()
            delta = low @ self.lorma_B.t()
            delta = delta * self.scale
            return z + delta.to(z.dtype)
        # perm: build R from B A, then add z (R - I)^T
        with torch.autocast(device_type=z.device.type, enabled=False):
            M = self.lorma_B @ self.lorma_A
            R = rotate_rows(M)
            D = R - get_eye(self.d, M.device)
        return z + z @ D.t()


def set_submodule(root, dotted_name, new_module):
    parts = dotted_name.split(".")
    parent = root
    for part in parts[:-1]:
        parent = getattr(parent, part)
    setattr(parent, parts[-1], new_module)


# Targets: attention q/k/v/o and gate/up/down of every MoE expert (routed and shared).
# The router and anything else inside the blocks stays frozen.
TARGET_NAMES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
ADAPTER_TARGETS = []
SKIPPED_LINEARS = {}
for name, module in LM.named_modules():
    if not isinstance(module, nn.Linear):
        continue
    if not name.startswith("blocks."):
        continue
    last = name.split(".")[-1]
    if last in TARGET_NAMES:
        ADAPTER_TARGETS.append(name)
    else:
        SKIPPED_LINEARS[last] = SKIPPED_LINEARS.get(last, 0) + 1
if len(ADAPTER_TARGETS) == 0:
    raise RuntimeError("No target linear layers found under blocks. Check the Model 4 module names.")
KIND_COUNT = {}
for name in ADAPTER_TARGETS:
    last = name.split(".")[-1]
    KIND_COUNT[last] = KIND_COUNT.get(last, 0) + 1
print("adapter target layers per kind:", KIND_COUNT)
print("linear layers inside the blocks that stay frozen (name: count):", SKIPPED_LINEARS)

with torch.no_grad():
    check_ids = torch.tensor([[5, 9, 12, 40, 77, SEP_ID]], device=DEVICE)
    before = backbone_hidden(check_ids).float().clone()

ADAPTER_MODULES = []
for name in ADAPTER_TARGETS:
    wrapped = LoRMALinear(LM.get_submodule(name), LORMA_R, LORMA_ALPHA, LORMA_DROPOUT, LORMA_VARIANT)
    set_submodule(LM, name, wrapped)
    ADAPTER_MODULES.append(wrapped)

with torch.no_grad():
    after = backbone_hidden(check_ids).float()
identity_diff = float((before - after).abs().max())
print("identity check (max abs diff at init, should be 0):", identity_diff)
assert identity_diff < 1e-4, "LoRMA changed the Phase-1 output at init"

ADAPTER_PARAMS = []
for m in ADAPTER_MODULES:
    ADAPTER_PARAMS.append(m.lorma_A)
    ADAPTER_PARAMS.append(m.lorma_B)
N_ADAPTER = sum(p.numel() for p in ADAPTER_PARAMS)
print("trainable LoRMA params: %s (%.2f%% of the model)" % (f"{N_ADAPTER:,}", 100.0 * N_ADAPTER / N_BASE))


def adapter_state():
    state = {}
    for name, m in zip(ADAPTER_TARGETS, ADAPTER_MODULES):
        state[name] = (m.lorma_A.detach().cpu().clone(), m.lorma_B.detach().cpu().clone())
    return state


def load_adapter_state(state):
    for name, m in zip(ADAPTER_TARGETS, ADAPTER_MODULES):
        a, b = state[name]
        with torch.no_grad():
            m.lorma_A.copy_(a.to(m.lorma_A.device))
            m.lorma_B.copy_(b.to(m.lorma_B.device))

## 9. Training: multi-positive contrastive loss
Per step: a batch of queries, a few sampled positives and BM25 hard negatives each, plus every other document in the step as an in-batch negative. A document that is gold for a query is never a negative for it (and neither is the query itself). The loss for a positive `p` of query `q` is `-log( e^{s(q,p)/t} / ( e^{s(q,p)/t} + sum over negatives e^{s(q,n)/t} ) )`, averaged over the sampled positives, so positives do not compete with each other.

In [ ]:
def set_train_mode():
    LM.eval()                      # Phase-1 dropout stays off
    for m in ADAPTER_MODULES:
        m.train()                  # adapter dropout on


def set_eval_mode():
    LM.eval()
    for m in ADAPTER_MODULES:
        m.eval()


# Activity checks. In a Mixture-of-Experts model the adapters only help if the model code really runs the wrapped layers.
def kind_of(name):
    return name.split(".")[-1]


for m in ADAPTER_MODULES:
    m.calls = 0
set_eval_mode()
with torch.no_grad():
    act_ids = torch.randint(3, LM.config.vocab_size, (4, 256), device=DEVICE)
    act_before = backbone_hidden(act_ids).float().clone()
calls_by_kind = {}
for name, m in zip(ADAPTER_TARGETS, ADAPTER_MODULES):
    k = kind_of(name)
    calls_by_kind[k] = calls_by_kind.get(k, 0) + m.calls
print("adapter calls per kind in one forward pass:", calls_by_kind)
for k in KIND_COUNT:
    if calls_by_kind.get(k, 0) == 0:
        raise RuntimeError("Adapters of kind %s were never called: the model bypasses the wrapped layers." % k)

# a random adapter must change the output
saved_state = adapter_state()
with torch.no_grad():
    for m in ADAPTER_MODULES:
        m.lorma_A.add_(0.05 * torch.randn_like(m.lorma_A))
        m.lorma_B.add_(0.05 * torch.randn_like(m.lorma_B))
    act_after = backbone_hidden(act_ids).float()
change = float((act_before - act_after).abs().max())
print("output change with a random adapter (should be > 0):", change)
if change <= 0:
    raise RuntimeError("A random adapter did not change the output: the adapters are not wired in.")
load_adapter_state(saved_state)

# every kind must receive a gradient through the checkpointed backbone
set_train_mode()
for p in ADAPTER_PARAMS:
    p.grad = None
act_loss = backbone_hidden(act_ids).float().pow(2).mean()
act_loss.backward()
grad_by_kind = {}
for name, m in zip(ADAPTER_TARGETS, ADAPTER_MODULES):
    k = kind_of(name)
    g = 0.0
    for p in (m.lorma_A, m.lorma_B):
        if p.grad is not None:
            g += float(p.grad.abs().sum())
    grad_by_kind[k] = grad_by_kind.get(k, 0.0) + g
print("gradient mass per kind:", {k: round(v, 6) for k, v in grad_by_kind.items()})
for k in KIND_COUNT:
    if grad_by_kind.get(k, 0.0) <= 0:
        raise RuntimeError("Adapters of kind %s got no gradient through the backbone." % k)
for p in ADAPTER_PARAMS:
    p.grad = None
set_eval_mode()


# training items: (split, query id)
TRAIN_SPLITS = ["train"]
if TRAIN_ON_DEV and "dev" in B["splits"]:
    TRAIN_SPLITS.append("dev")
TRAIN_ITEMS = []
for split in TRAIN_SPLITS:
    for qid in B["query_ids"][split]:
        if len(GOLD[split][qid]) > 0:
            TRAIN_ITEMS.append((split, qid))
if SMOKE_TEST:
    TRAIN_ITEMS = TRAIN_ITEMS[:SMOKE_TRAIN_QUERIES]
print("training queries:", len(TRAIN_ITEMS), "| splits:", TRAIN_SPLITS)

HARD_NEG = B.get("hard_neg", {})


def sample_query(item):
    split, qid = item
    gold = GOLD[split][qid]
    pos = random.sample(gold, min(POS_PER_QUERY, len(gold)))
    lo = HARD_NEG_RANGE[0]
    hi = HARD_NEG_RANGE[1]
    pool_hard = []
    if split in HARD_NEG and qid in HARD_NEG[split]:
        pool_hard = HARD_NEG[split][qid][lo:hi]
    negs = []
    if len(pool_hard) > 0:
        negs = random.sample(pool_hard, min(HARD_NEG_PER_QUERY, len(pool_hard)))
    gold_set = GOLDSET[(split, qid)]
    tries = 0
    pool_ids = B["pool_ids"][split]
    while len(negs) < HARD_NEG_PER_QUERY and tries < 50:
        tries += 1
        c = random.choice(pool_ids)
        if c in gold_set or c == qid or c in negs:
            continue
        negs.append(c)
    return pos, negs


def step_loss(items):
    q_specs = []
    cand_specs = []
    cand_index = {}
    positives = []
    for item in items:
        split, qid = item
        pos, negs = sample_query(item)
        q_specs.append((split, "q", qid))
        positives.append((split, pos))
        for c in pos + negs:
            spec = (split, "c", c)
            if spec not in cand_index:
                cand_index[spec] = len(cand_specs)
                cand_specs.append(spec)
    all_specs = q_specs + cand_specs
    E = embed_docs(all_specs, True)
    Qv = E[:len(q_specs)]
    Cv = E[len(q_specs):]
    sims = (Qv @ Cv.t()) / TEMPERATURE
    n_q = len(items)
    n_c = len(cand_specs)
    neg_mask = torch.ones(n_q, n_c, dtype=torch.bool)
    pos_mask = torch.zeros(n_q, n_c, dtype=torch.bool)
    for i, item in enumerate(items):
        split, qid = item
        gold_set = GOLDSET[(split, qid)]
        for j, spec in enumerate(cand_specs):
            c_id = spec[2]
            if c_id in gold_set or c_id == qid:
                neg_mask[i, j] = False
        for c in positives[i][1]:
            pos_mask[i, cand_index[(split, "c", c)]] = True
    neg_mask = neg_mask.to(sims.device)
    pos_mask = pos_mask.to(sims.device)
    neg_lse = torch.logsumexp(sims.masked_fill(~neg_mask, -1e9), dim=1, keepdim=True)
    terms = F.softplus(neg_lse - sims)
    per_query = (terms * pos_mask.float()).sum(1) / pos_mask.float().sum(1).clamp(min=1.0)
    return per_query.mean()


TRAIN_PARAMS = list(ADAPTER_PARAMS)
PROJ_PARAMS = []
if PROJ_MODULE is not None:
    PROJ_PARAMS = list(PROJ_MODULE.parameters())

CKPT_PATH = os.path.join(OUT_DIR, "ckpt_pcr.pt")
BEST_PATH = os.path.join(OUT_DIR, "adapter_best.pt")
STOPPED_EARLY = False
TRAIN_SECONDS = 0.0
HISTORY = []
BEST_F1 = -1.0
BEST_EPOCH = -1
BEST_STATE = None


def current_state_dict():
    proj_state = None
    if PROJ_MODULE is not None:
        proj_state = {}
        for k, v in PROJ_MODULE.state_dict().items():
            proj_state[k] = v.detach().cpu().clone()
    return {"adapter": adapter_state(), "proj": proj_state}


def load_state_dict_all(blob):
    load_adapter_state(blob["adapter"])
    if PROJ_MODULE is not None and blob["proj"] is not None:
        PROJ_MODULE.load_state_dict(blob["proj"])


def quick_dev_f1():
    set_eval_mode()
    qids, pids, Qe, Pe = embed_split("dev")
    p, r, f1 = plain_f1(Qe, Pe, qids, pids, GOLD["dev"])
    set_train_mode()
    return p, r, f1


if RUN_TRAIN and len(TRAIN_ITEMS) > 0:
    steps_per_epoch = int(math.ceil(len(TRAIN_ITEMS) / QUERIES_PER_STEP))
    total_steps = EPOCHS * steps_per_epoch
    warm_steps = max(1, int(WARMUP_FRAC * total_steps))
    print("steps per epoch:", steps_per_epoch, "| total steps:", total_steps)

    groups = [{"params": TRAIN_PARAMS, "lr": LR}]
    if len(PROJ_PARAMS) > 0:
        groups.append({"params": PROJ_PARAMS, "lr": PROJ_LR})
    optimizer = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
    base_lrs = []
    for g in optimizer.param_groups:
        base_lrs.append(g["lr"])
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)

    def lr_scale(step):
        if step < warm_steps:
            return (step + 1) / warm_steps
        progress = (step - warm_steps) / max(1, total_steps - warm_steps)
        return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))

    state = {"epoch": 0, "pos": 0, "step": 0, "seconds": 0.0, "perm": None}

    if RESUME:
        found = None
        if os.path.exists(CKPT_PATH):
            found = CKPT_PATH
        else:
            found = find_one_file(["/kaggle/input"], "ckpt_pcr.pt")
        if found is not None:
            blob = torch.load(found, map_location="cpu", weights_only=False)
            load_state_dict_all(blob["model"])
            optimizer.load_state_dict(blob["opt"])
            scaler.load_state_dict(blob["scaler"])
            state = blob["state"]
            HISTORY = blob["history"]
            BEST_F1 = blob["best_f1"]
            BEST_EPOCH = blob["best_epoch"]
            BEST_STATE = blob["best_state"]
            if found != CKPT_PATH:
                shutil.copy(found, CKPT_PATH)
            print("resumed from", found, "| epoch", state["epoch"], "| step", state["step"])

    def save_ckpt():
        blob = {"model": current_state_dict(), "opt": optimizer.state_dict(), "scaler": scaler.state_dict(),
                "state": state, "history": HISTORY, "best_f1": BEST_F1, "best_epoch": BEST_EPOCH,
                "best_state": BEST_STATE}
        torch.save(blob, CKPT_PATH + ".tmp")
        os.replace(CKPT_PATH + ".tmp", CKPT_PATH)

    seconds_before = state["seconds"]
    t_run = time.time()
    t_last_log = time.time()
    run_loss = 0.0
    run_n = 0
    set_train_mode()

    for epoch in range(state["epoch"], EPOCHS):
        if state["perm"] is None:
            perm = list(range(len(TRAIN_ITEMS)))
            random.shuffle(perm)
            state["perm"] = perm
            state["pos"] = 0
        perm = state["perm"]
        pos = state["pos"]
        while pos < len(perm):
            batch_idx = perm[pos:pos + QUERIES_PER_STEP]
            items = []
            for i in batch_idx:
                items.append(TRAIN_ITEMS[i])
            scale = lr_scale(state["step"])
            for g, base_lr in zip(optimizer.param_groups, base_lrs):
                g["lr"] = base_lr * scale
            loss = step_loss(items)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(TRAIN_PARAMS + PROJ_PARAMS, GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

            pos += len(batch_idx)
            state["pos"] = pos
            state["step"] += 1
            run_loss += float(loss.detach())
            run_n += 1

            if state["step"] % LOG_EVERY_STEPS == 0:
                elapsed = time.time() - t_last_log
                per_step = elapsed / LOG_EVERY_STEPS
                left = total_steps - state["step"]
                print("epoch %d | step %d/%d | loss %.4f | lr %.2e | %.1f s/step | eta %.0f min" % (
                    epoch, state["step"], total_steps, run_loss / max(run_n, 1), LR * scale,
                    per_step, left * per_step / 60.0))
                run_loss = 0.0
                run_n = 0
                t_last_log = time.time()

            state["seconds"] = seconds_before + (time.time() - t_run)
            if state["step"] % SAVE_EVERY_STEPS == 0:
                save_ckpt()
            if time.time() - START_TIME > SESSION_CAP_SEC:
                save_ckpt()
                STOPPED_EARLY = True
                print("Session time cap reached. Checkpoint saved. Attach this notebook's output as an input and run again to resume.")
                break
        if STOPPED_EARLY:
            break

        # end of epoch: validate on dev
        state["epoch"] = epoch + 1
        state["pos"] = 0
        state["perm"] = None
        entry = {"epoch": epoch, "step": state["step"]}
        if not TRAIN_ON_DEV and "dev" in B["splits"]:
            t_eval = time.time()
            p, r, f1 = quick_dev_f1()
            entry["dev_f1"] = f1
            entry["dev_precision"] = p
            entry["dev_recall"] = r
            print("epoch %d finished | dev Micro-F1@%d = %.4f (P %.4f, R %.4f) | eval %.0fs" % (epoch, TOP_K, f1, p, r, time.time() - t_eval))
            if f1 > BEST_F1:
                BEST_F1 = f1
                BEST_EPOCH = epoch
                BEST_STATE = current_state_dict()
                torch.save(BEST_STATE, BEST_PATH)
                print("  new best on dev")
        else:
            print("epoch", epoch, "finished")
        HISTORY.append(entry)
        save_ckpt()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    TRAIN_SECONDS = state["seconds"]
    if not STOPPED_EARLY:
        print("training done | %.1f min | best dev F1 %.4f at epoch %d" % (TRAIN_SECONDS / 60.0, BEST_F1, BEST_EPOCH))
        if BEST_STATE is None or TRAIN_ON_DEV:
            BEST_STATE = current_state_dict()
            torch.save(BEST_STATE, BEST_PATH)
else:
    if os.path.exists(BEST_PATH):
        BEST_STATE = torch.load(BEST_PATH, map_location="cpu", weights_only=False)
    else:
        found = find_one_file(["/kaggle/input"], "adapter_best.pt")
        if found is not None:
            BEST_STATE = torch.load(found, map_location="cpu", weights_only=False)
    if BEST_STATE is not None:
        print("loaded saved adapter weights")
    else:
        print("No training and no saved adapter weights: the model is the plain Phase-1 model.")

if not STOPPED_EARLY and BEST_STATE is not None:
    load_state_dict_all(BEST_STATE)
    print("best adapter loaded (dev F1 %.4f at epoch %d)" % (BEST_F1, BEST_EPOCH))
set_eval_mode()

## 10. Dev: sweep the inference options, choose the best, analyse errors

In [ ]:
CHOSEN = {"center": False, "norm": "none", "param": 0, "lambda": 0.0}
SWEEP_ROWS = []
DEV_REPORT = {}

if STOPPED_EARLY:
    print("Training stopped at the session cap: attach this output as an input and re-run to continue. Skipping evaluation.")
elif "dev" in B["splits"]:
    t0 = time.time()
    qids, pids, Qe, Pe = embed_split("dev")
    dev_gold = GOLD["dev"]
    dev_bm25 = bm25_scores("dev", qids, pids)
    SWEEP_ROWS = run_sweep(Qe, Pe, qids, pids, dev_gold, dev_bm25)
    CHOSEN = {"center": SWEEP_ROWS[0]["center"], "norm": SWEEP_ROWS[0]["norm"],
              "param": SWEEP_ROWS[0]["param"], "lambda": SWEEP_ROWS[0]["lambda"]}

    p_plain, r_plain, f1_plain = plain_f1(Qe, Pe, qids, pids, dev_gold)
    print("dev Micro-F1@%d, plain cosine (neural only): %.4f (P %.4f, R %.4f)" % (TOP_K, f1_plain, p_plain, r_plain))
    print("top 8 inference options on dev:")
    for row in SWEEP_ROWS[:8]:
        print("  center=%-5s norm=%-4s param=%-5s bm25_weight=%-4s -> F1 %.4f (P %.4f, R %.4f)" % (
            row["center"], row["norm"], row["param"], row["lambda"], row["f1"], row["precision"], row["recall"]))
    print("chosen:", CHOSEN, "-> dev F1 %.4f" % SWEEP_ROWS[0]["f1"])
    neural_only = []
    for row in SWEEP_ROWS:
        if row["lambda"] == 0.0:
            neural_only.append(row)
    print("best neural-only option: center=%s norm=%s param=%s -> F1 %.4f" % (
        neural_only[0]["center"], neural_only[0]["norm"], neural_only[0]["param"], neural_only[0]["f1"]))
    print("note: the options were picked on dev (%d queries), so dev F1 is slightly optimistic." % len(qids))

    with open(os.path.join(OUT_DIR, "dev_sweep.json"), "w") as f:
        json.dump(SWEEP_ROWS, f)

    # error analysis with the chosen options
    S_final = final_scores(Qe, Pe, qids, pids, CHOSEN, dev_bm25)
    lists = top_ids(S_final, pids, TOP_K)
    dev_lists_100 = top_ids(S_final, pids, 100)
    DEV_REPORT["recall_at"] = {}
    for k in [7, 20, 50, 100]:
        DEV_REPORT["recall_at"][str(k)] = recall_at(S_final, qids, pids, dev_gold, k)
    print("dev recall@k (chosen options):", DEV_REPORT["recall_at"])

    q_len = []
    for q in qids:
        q_len.append(len(B["tokens"]["dev"]["queries"][q]))
    q_len = np.array(q_len)
    cuts = np.percentile(q_len, [33, 66])
    buckets = {"short query": [], "medium query": [], "long query": []}
    gold_buckets = {"1 gold": [], "2-4 gold": [], "5-7 gold": [], ">7 gold": []}
    zero_hit = 0
    for r, q in enumerate(qids):
        g = set(dev_gold[q])
        hits = 0
        for c in lists[r]:
            if c in g:
                hits += 1
        if hits == 0:
            zero_hit += 1
        if q_len[r] <= cuts[0]:
            buckets["short query"].append((hits, len(g)))
        elif q_len[r] <= cuts[1]:
            buckets["medium query"].append((hits, len(g)))
        else:
            buckets["long query"].append((hits, len(g)))
        if len(g) == 1:
            gold_buckets["1 gold"].append((hits, len(g)))
        elif len(g) <= 4:
            gold_buckets["2-4 gold"].append((hits, len(g)))
        elif len(g) <= 7:
            gold_buckets["5-7 gold"].append((hits, len(g)))
        else:
            gold_buckets[">7 gold"].append((hits, len(g)))
    DEV_REPORT["zero_hit_queries"] = zero_hit
    DEV_REPORT["by_query_length"] = {}
    DEV_REPORT["by_gold_count"] = {}
    print("queries with no hit in the top %d: %d / %d" % (TOP_K, zero_hit, len(qids)))
    for title, table, key in [("query length", buckets, "by_query_length"), ("gold count", gold_buckets, "by_gold_count")]:
        for name in table:
            rows = table[name]
            if len(rows) == 0:
                continue
            tp = sum(h for h, g in rows)
            n_gold = sum(g for h, g in rows)
            prec = tp / (TOP_K * len(rows))
            rec = tp / max(n_gold, 1)
            DEV_REPORT[key][name] = {"queries": len(rows), "precision": prec, "recall": rec}
            print("  %-13s %-13s n=%3d  P@%d %.3f  R@%d %.3f" % (title, name, len(rows), TOP_K, prec, TOP_K, rec))

    with open(os.path.join(OUT_DIR, "dev_predictions.json"), "w") as f:
        out = {}
        for r, q in enumerate(qids):
            out[q] = lists[r]
        json.dump(out, f)
    print("dev analysis took %.0fs" % (time.time() - t0))
else:
    print("No dev split: using plain cosine for the test predictions.")

## 11. Test predictions
Exactly 7 unique ids per test query, taken only from the test pool, ordered best first. The query itself is never returned.

In [ ]:
TEST_SECONDS = 0.0
PRED_PATH = os.path.join(OUT_DIR, "prediction.json")
if STOPPED_EARLY:
    print("Skipped (training not finished).")
elif RUN_TEST and "test" in B["splits"]:
    t0 = time.time()
    qids, pids, Qe, Pe = embed_split("test")
    test_bm25 = bm25_scores("test", qids, pids)
    TEST_SECONDS = time.time() - t0

    S_final = final_scores(Qe, Pe, qids, pids, CHOSEN, test_bm25)
    final_lists = top_ids(S_final, pids, TOP_K)
    S_plain = make_scores(Qe, Pe, qids, pids, False)
    plain_lists = top_ids(S_plain, pids, TOP_K)

    def build_prediction(lists):
        pred = {}
        pool_set = set(pids)
        for r, q in enumerate(qids):
            ranked = lists[r]
            if len(ranked) != TOP_K or len(set(ranked)) != TOP_K:
                raise RuntimeError("bad ranking for query " + q)
            for c in ranked:
                if c not in pool_set:
                    raise RuntimeError("candidate outside the test pool for query " + q)
                if c == q:
                    raise RuntimeError("query retrieved itself: " + q)
            pred[q] = ranked
        if len(pred) != len(B["query_ids"]["test"]):
            raise RuntimeError("missing test queries")
        return pred

    prediction = build_prediction(final_lists)
    with open(PRED_PATH, "w") as f:
        json.dump(prediction, f)
    with open(os.path.join(OUT_DIR, "prediction_plain_cosine.json"), "w") as f:
        json.dump(build_prediction(plain_lists), f)
    zip_path = os.path.join(OUT_DIR, "submission.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(PRED_PATH, "prediction.json")
    print("wrote prediction.json for %d test queries (chosen options: %s)" % (len(prediction), CHOSEN))
    print("wrote prediction_plain_cosine.json (neural cosine only) and submission.zip")
    print("test embedding + scoring: %.0fs for %d queries and %d pool docs" % (TEST_SECONDS, len(qids), len(pids)))
else:
    print("RUN_TEST is off or there is no test split.")

## 12. Run summary (for the report)

In [ ]:
summary = {
    "model": "Phase-1 Model 4 (GQA+MHA+MoE)",
    "adapter": "LoRMA",
    "adapter_targets_by_kind": KIND_COUNT,
    "adapter_settings": {"variant": LORMA_VARIANT, "r": LORMA_R, "alpha": LORMA_ALPHA, "dropout": LORMA_DROPOUT},
    "base_params": int(N_BASE),
    "adapter_params": int(N_ADAPTER),
    "projection_params": int(D_MODEL * D_MODEL) if PROJ_MODULE is not None else 0,
    "embedding_dim": int(D_MODEL),
    "doc_view": DOC_VIEW,
    "head_chunks": HEAD_CHUNKS,
    "tail_chunks": TAIL_CHUNKS,
    "chunk_len": CHUNK_LEN,
    "pooling": POOLING,
    "temperature": TEMPERATURE,
    "queries_per_step": QUERIES_PER_STEP,
    "pos_per_query": POS_PER_QUERY,
    "hard_neg_per_query": HARD_NEG_PER_QUERY,
    "hard_neg_range": list(HARD_NEG_RANGE),
    "epochs": EPOCHS,
    "lr": LR,
    "train_seconds": TRAIN_SECONDS,
    "stopped_early": STOPPED_EARLY,
    "history": HISTORY,
    "best_dev_f1_plain_cosine": BEST_F1,
    "best_epoch": BEST_EPOCH,
    "zero_shot_dev": ZERO_SHOT,
    "chosen_inference_options": CHOSEN,
    "dev_best_sweep_row": SWEEP_ROWS[0] if len(SWEEP_ROWS) > 0 else None,
    "dev_report": DEV_REPORT,
    "test_embed_seconds": TEST_SECONDS,
    "peak_gpu_mem_gb": (torch.cuda.max_memory_allocated() / 1e9) if DEVICE.type == "cuda" else None,
}
with open(os.path.join(OUT_DIR, "run_summary.json"), "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(json.dumps(summary, indent=2, default=str)[:3000])
print("\nfiles in", OUT_DIR)
for name in sorted(os.listdir(OUT_DIR)):
    full = os.path.join(OUT_DIR, name)
    if os.path.isfile(full):
        print("  %-30s %.2f MB" % (name, os.path.getsize(full) / 1e6))